In [ ]:
import os
import sys
import gc
import json
import math
import random
import re
from dataclasses import dataclass, asdict
from collections import Counter

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

try:
    from peft import PeftModel
    HAS_PEFT = True
except Exception:
    HAS_PEFT = False

# ============================================================
# CONFIG
# ============================================================
BASE_MODEL = "microsoft/phi-3-mini-4k-instruct"
SFT_PATH = "/kaggle/input/datasets/adithyaled24b039/phi3-sft-folderr/phi3_sft_lora"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32

SEED = 42
EVAL_SEED = 42

N_GSM8K = 18
N_STRATEGYQA = 18
N_MMLU_PER_SUBJECT = 6
MMLU_SUBJECTS = ["abstract_algebra", "college_mathematics", "logical_fallacies"]

@dataclass(frozen=True)
class PromptRecipe:
    name: str
    prefix: str
    order: str
    style: str
    spacing: str
    strength: float
    note: str

PROMPT_RECIPES = [
    PromptRecipe("baseline_plain", "",   "q_i_a",     "plain",      "compact", 0.00, "Minimal prompt"),
    PromptRecipe("hash_plain",     "##", "q_i_a",     "plain",      "compact", 0.15, "Single delimiter"),
    PromptRecipe("hash_roomy",     "##", "q_i_a",     "plain",      "roomy",   0.25, "Delimiter + whitespace"),
    PromptRecipe("hash_think",     "##", "q_i_t_a",   "think",      "roomy",   0.45, "Think then answer"),
    PromptRecipe("hash_xml_think", "##", "q_i_t_a",   "xml_think",  "roomy",   0.70, "Think + XML answer"),
    PromptRecipe("hash_selfcheck",  "##", "q_i_t_a",   "self_check", "roomy",   0.85, "Think + verify"),
    PromptRecipe("answer_first",    "",   "a_q_i",     "plain",      "compact", 0.35, "Answer directive first"),
    PromptRecipe("answer_first_xml", "##", "a_q_i",     "xml_think",  "roomy",   0.80, "Answer-first XML"),
    PromptRecipe("verbose_xml",     "###", "i_q_t_a",   "xml_think",  "spacious",1.00, "Verbose XML scaffold"),
]
BASELINE_RECIPE = PROMPT_RECIPES[0].name

MAX_NEW = {"gsm8k": 128, "strategyqa": 24, "mmlu": 16}
TOP_K_TOKENS = 5
PREFIX_ATTN_WINDOW = 8

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

TOKENIZER = None
MODEL = None
BLOCKS = None
FINAL_NORM = None
LM_HEAD = None
MODEL_LAYERS = None
MODEL_HIDDEN = None

# ============================================================
# UTILITIES
# ============================================================
def free_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def to_jsonable(x):
    try: return json.dumps(x, ensure_ascii=False)
    except: return json.dumps(str(x), ensure_ascii=False)

def normalize_number(x):
    if x is None: return None
    try: return float(re.sub(r"[$,]", "", str(x)))
    except: return None

def canonical_number_str(x):
    v = normalize_number(x)
    if v is None: return None
    if abs(v - round(v)) < 1e-6: return str(int(round(v)))
    return f"{v:.6f}".rstrip("0").rstrip(".")

def same_numeric(a, b, tol=1e-6):
    na = normalize_number(a)
    nb = normalize_number(b)
    if na is None or nb is None: return False
    return abs(na - nb) <= tol

def softmax_np(x):
    x = np.asarray(x, dtype=np.float64)
    x = x - np.max(x)
    ex = np.exp(x)
    return ex / (ex.sum() + 1e-12)

def entropy_from_probs(p):
    p = np.asarray(p, dtype=np.float64)
    p = p[p > 0]
    if len(p) == 0: return float("nan")
    return float(-np.sum(p * np.log(p + 1e-12)))

def group_logsumexp(logits, ids):
    if len(ids) == 0: return float("-inf")
    vals = torch.tensor([float(logits[i].item()) for i in ids], device=logits.device)
    return float(torch.logsumexp(vals, dim=0).item())

# ============================================================
# PARSING
# ============================================================
def extract_number(text):
    if text is None: return None
    m = re.findall(r"<answer>(.*?)</answer>", text, flags=re.IGNORECASE | re.DOTALL)
    span = m[-1] if m else text
    span = re.sub(r"[$,]", "", span)
    nums = re.findall(r"-?\d+\.?\d*", span)
    if nums: return nums[-1]
    nums = re.findall(r"-?\d+\.?\d*", re.sub(r"[$,]", "", text))
    return nums[-1] if nums else None

def extract_yes_no(text):
    if text is None: return None
    m = re.findall(r"<answer>(.*?)</answer>", text, flags=re.IGNORECASE | re.DOTALL)
    span = m[-1] if m else text
    hits = re.findall(r"\b(yes|no)\b", span, flags=re.IGNORECASE)
    if hits: return hits[-1].lower()
    hits = re.findall(r"\b(yes|no)\b", text, flags=re.IGNORECASE)
    return hits[-1].lower() if hits else None

def extract_mcq(text, choices=None):
    if text is None: return None
    span = text
    m = re.findall(r"<answer>(.*?)</answer>", text, flags=re.IGNORECASE | re.DOTALL)
    if m: span = m[-1]
    span_up = span.upper().strip()
    if span_up in ("A", "B", "C", "D"): return span_up
    patterns = [r"ANSWER\s*[:\-]?\s*\(?([ABCD])\)?", r"\b([ABCD])\b\s*$", r"<ANSWER>\s*\(?([ABCD])\)?"]
    for pat in patterns:
        hits = re.findall(pat, span_up)
        if hits: return hits[-1].upper()
    hits = re.findall(r"\b([ABCD])\b", span_up)
    if hits: return hits[-1].upper()
    if choices is not None:
        low = span.lower()
        for i, c in enumerate(choices):
            if str(c).strip().lower() in low: return chr(65 + i)
    return None

def target_prediction(task, completion, choices=None):
    if task == "gsm8k": return canonical_number_str(extract_number(completion))
    if task == "strategyqa": return extract_yes_no(completion)
    if task == "mmlu": return extract_mcq(completion, choices=choices)
    raise ValueError(task)

def exact_correct(task, pred, gold):
    if task == "gsm8k": return same_numeric(pred, gold)
    return pred == gold

def token_ids_for_text(tokenizer, text):
    forms = [text, f" {text}", f"\n{text}", text.upper(), f" {text.upper()}", text.lower(), f" {text.lower()}"]
    ids = []
    for form in forms:
        enc = tokenizer.encode(form, add_special_tokens=False)
        if len(enc) == 1: ids.append(enc[0])
    if not ids:
        enc = tokenizer.encode(text, add_special_tokens=False)
        if len(enc) > 0: ids.append(enc[0])
    return sorted(set(ids))

# ============================================================
# MODEL LOADING
# ============================================================
def load_model():
    global TOKENIZER, MODEL, BLOCKS, FINAL_NORM, LM_HEAD, MODEL_LAYERS, MODEL_HIDDEN
    
    src = SFT_PATH if (HAS_PEFT and os.path.exists(SFT_PATH)) else BASE_MODEL
    TOKENIZER = AutoTokenizer.from_pretrained(src)
    if TOKENIZER.pad_token is None: TOKENIZER.pad_token = TOKENIZER.eos_token

    kwargs = {"torch_dtype": DTYPE, "attn_implementation": "eager"}
    try:
        model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **kwargs)
    except Exception:
        kwargs.pop("attn_implementation", None)
        model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **kwargs)

    model = model.to(DEVICE)
    model.eval()

    if HAS_PEFT and os.path.exists(SFT_PATH):
        try:
            model = PeftModel.from_pretrained(model, SFT_PATH).merge_and_unload().to(DEVICE)
            model.eval()
        except: pass

    model.config.use_cache = False
    MODEL = model
    
    # Mechanics Discovery
    BLOCKS = getattr(getattr(model, "model", None), "layers", None)
    FINAL_NORM = getattr(getattr(model, "model", None), "norm", None)
    LM_HEAD = getattr(model, "lm_head", None)
    
    MODEL_LAYERS = len(BLOCKS)
    MODEL_HIDDEN = int(LM_HEAD.weight.shape[1])
    print(f"Model loaded: {MODEL_LAYERS} Layers, {MODEL_HIDDEN} Hidden Dim", flush=True)

# ============================================================
# PROMPT ENGINE
# ============================================================
def task_instruction(task, recipe):
    if task == "gsm8k":
        base = "Solve the math problem carefully. Give the final numeric answer only."
        if recipe.style in ("think", "xml_think", "self_check"): base = "Solve the math problem carefully and show a short reasoning chain."
        if recipe.style == "self_check": base += " Then verify the arithmetic before answering."
        return base
    if task == "strategyqa":
        base = "Decide whether the answer is yes or no."
        if recipe.style in ("think", "xml_think", "self_check"): base = "Reason carefully and decide whether the answer is yes or no."
        if recipe.style == "self_check": base += " Then verify the decision before answering."
        return base
    if task == "mmlu":
        base = "Choose the single best answer from A, B, C, or D."
        if recipe.style in ("think", "xml_think", "self_check"): base = "Reason about the options and choose the single best answer from A, B, C, or D."
        if recipe.style == "self_check": base += " Then verify that the chosen letter matches the content."
        return base
    raise ValueError(task)

def reasoning_directive(recipe):
    if recipe.style == "plain": return ""
    if recipe.style == "think": return "Think step by step before answering."
    if recipe.style == "xml_think": return "Think step by step inside <think></think>."
    if recipe.style == "self_check": return "Think step by step, then check for mistakes before finalizing your answer."
    return ""

def answer_directive(task, recipe):
    if task == "gsm8k":
        if recipe.style in ("xml_think", "self_check", "answer_first_xml", "verbose_xml"): return "Put ONLY the final numeric answer inside <answer></answer>."
        return "Answer:"
    if task == "strategyqa":
        if recipe.style in ("xml_think", "self_check", "answer_first_xml", "verbose_xml"): return "Put ONLY yes or no inside <answer></answer>."
        return "Answer:"
    if task == "mmlu":
        if recipe.style in ("xml_think", "self_check", "answer_first_xml", "verbose_xml"): return "Put ONLY one letter (A, B, C, or D) inside <answer></answer>."
        return "Answer:"
    raise ValueError(task)

def build_prompt(sample, recipe):
    task = sample["task"]
    q_header = f"{recipe.prefix} Question:" if recipe.prefix else "Question:"
    
    question_block = f"{q_header} {sample['question']}"
    if task == "mmlu":
        options = "\n".join([f"{chr(65+i)}. {c}" for i, c in enumerate(sample["choices"])])
        question_block = f"{question_block}\n{options}"

    i_block = task_instruction(task, recipe)
    r_block = reasoning_directive(recipe)
    a_block = answer_directive(task, recipe)

    if recipe.order == "q_i_a": blocks = [question_block, i_block, r_block, a_block]
    elif recipe.order == "q_i_t_a": blocks = [question_block, i_block, r_block, a_block]
    elif recipe.order == "i_q_t_a": blocks = [i_block, question_block, r_block, a_block]
    elif recipe.order == "a_q_i": blocks = [a_block, question_block, i_block, r_block]
    else: blocks = [question_block, a_block, i_block, r_block]

    sep = "\n\n" if recipe.spacing in ("roomy", "spacious") else "\n"
    return sep.join([b for b in blocks if b and str(b).strip()])

def task_candidate_groups(sample):
    if sample["task"] == "strategyqa": return [token_ids_for_text(TOKENIZER, "yes"), token_ids_for_text(TOKENIZER, "no")], 0 if sample["gold"] == "yes" else 1
    if sample["task"] == "mmlu": return [token_ids_for_text(TOKENIZER, x) for x in ["A", "B", "C", "D"]], ord(sample["gold"]) - 65
    return None, None

# ============================================================
# ANALYSIS ENGINE
# ============================================================
@torch.inference_mode()
def greedy_generate(prompt, max_new_tokens):
    inputs = TOKENIZER(prompt, return_tensors="pt").to(DEVICE)
    out = MODEL.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=TOKENIZER.eos_token_id)
    return TOKENIZER.decode(out[0], skip_special_tokens=True), TOKENIZER.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

def analyze_prompt(prompt, target_text, candidate_groups=None, target_group_idx=None):
    inputs = TOKENIZER(prompt, return_tensors="pt").to(DEVICE)
    with torch.inference_mode():
        out = MODEL(**inputs, output_hidden_states=True, output_attentions=True, use_cache=False, return_dict=True)

    logits = out.logits[0, -1].float()
    log_probs = torch.log_softmax(logits, dim=-1)
    full_entropy = float((-torch.softmax(logits, dim=-1) * log_probs).sum().item())

    target_ids = token_ids_for_text(TOKENIZER, target_text)
    
    # Margin
    t_vals = [float(logits[i].item()) for i in target_ids]
    target_logit = max(t_vals) if t_vals else float(logits[int(TOKENIZER.eos_token_id)].item())
    target_id = target_ids[int(np.argmax(t_vals))] if t_vals else int(TOKENIZER.eos_token_id)
    
    masked = logits.clone()
    masked[target_ids] = float("-inf")
    target_margin = float(target_logit - masked.max().item())

    # Candidate metrics
    candidate_metrics = None
    if candidate_groups is not None and target_group_idx is not None:
        scores = [group_logsumexp(logits, ids_) for ids_ in candidate_groups]
        g_probs = softmax_np(scores)
        candidate_metrics = {
            "group_entropy": entropy_from_probs(g_probs),
            "group_margin": float(scores[target_group_idx] - max([s for i, s in enumerate(scores) if i != target_group_idx])),
        }

    # Layerwise DLA
    target_weight = LM_HEAD.weight[target_id].float()
    dla_curve = []
    for h in out.hidden_states:
        vec = FINAL_NORM(h[:, -1, :]) if FINAL_NORM else h[:, -1, :]
        dla_curve.append(float(torch.dot(vec.squeeze(0).float(), target_weight).item()))

    # Prefix Attention
    attn_curve = []
    for layer_attn in out.attentions:
        a = layer_attn[0]
        qpos = a.shape[-2] - 1
        prefix_idx = list(range(min(PREFIX_ATTN_WINDOW, a.shape[-1])))
        if len(prefix_idx) == 0: attn_curve.append(float("nan"))
        else: attn_curve.append(float(a[:, qpos, prefix_idx].sum(dim=-1).mean().item()))

    return {
        "target_margin": target_margin,
        "full_entropy": full_entropy,
        "candidate_metrics": candidate_metrics,
        "dla_curve": dla_curve,
        "attention_curve": attn_curve,
        "n_prompt_tokens": int(inputs["input_ids"].shape[1])
    }

# ============================================================
# MAIN COMPUTE LOOP
# ============================================================
print("Loading model and tokenizer...", flush=True)
load_model()

# Create dummy samples
ds_gsm = list(load_dataset("gsm8k", "main", split="test").select(range(N_GSM8K)))
samples_gsm = [{"task": "gsm8k", "sample_id": i, "question": s["question"], "gold": canonical_number_str(s["answer"].split("####", 1)[1].strip())} for i,s in enumerate(ds_gsm)]

ds_sqa = list(load_dataset("ChilleD/StrategyQA", split="test").select(range(N_STRATEGYQA)))
samples_sqa = [{"task": "strategyqa", "sample_id": i, "question": s["question"], "gold": "yes" if bool(s["answer"]) else "no"} for i,s in enumerate(ds_sqa)]

ds_mmlu = list(load_dataset("cais/mmlu", MMLU_SUBJECTS[0], split="test").select(range(N_MMLU_PER_SUBJECT)))
samples_mmlu = [{"task": "mmlu", "sample_id": i, "subject": MMLU_SUBJECTS[0], "question": s["question"], "choices": list(s["choices"]), "gold": chr(65 + int(s["answer"]))} for i,s in enumerate(ds_mmlu)]

all_samples = samples_gsm + samples_sqa + samples_mmlu

all_rows = []
print(f"\n[ATLAS] Running Counterfactual Prompt Surgery on {len(all_samples)} samples...", flush=True)

for sample in all_samples:
    print(f"\nEvaluating Sample {sample['sample_id']} | Task: {sample['task']} | Q: {sample['question'][:50]}...", flush=True)
    c_groups, t_idx = task_candidate_groups(sample)
    content_target = sample["gold"]
    
    for r_idx, recipe in enumerate(PROMPT_RECIPES):
        print(f"  -> Recipe {r_idx+1}/{len(PROMPT_RECIPES)}: {recipe.name}", flush=True)
        prompt = build_prompt(sample, recipe)
        format_target = f"<answer>{content_target}</answer>" if recipe.style in ("xml_think", "self_check", "answer_first_xml", "verbose_xml") else content_target
        
        full_out, comp = greedy_generate(prompt, MAX_NEW[sample["task"]])
        pred = target_prediction(sample["task"], comp, sample.get("choices"))
        
        fmt_m = analyze_prompt(prompt, format_target)
        cnt_m = analyze_prompt(prompt, content_target, c_groups, t_idx)
        
        first_ids = TOKENIZER.encode(comp, add_special_tokens=False)
        first_id = int(first_ids[0]) if first_ids else None
        
        row = {
            "task": sample["task"], "sample_id": sample["sample_id"], "recipe": recipe.name, "prompt_strength": recipe.strength,
            "gold": sample["gold"], "prediction": pred, "correct": exact_correct(sample["task"], pred, sample["gold"]),
            "format_target_margin": fmt_m["target_margin"], "content_target_margin": cnt_m["target_margin"],
            "format_full_entropy": fmt_m["full_entropy"], "content_full_entropy": cnt_m["full_entropy"],
            "format_dla_curve": to_jsonable(fmt_m["dla_curve"]), "content_dla_curve": to_jsonable(cnt_m["dla_curve"]),
            "format_attention_curve": to_jsonable(fmt_m["attention_curve"]), "content_attention_curve": to_jsonable(cnt_m["attention_curve"]),
            "prompt_tokens": fmt_m["n_prompt_tokens"], "completion_tokens": len(first_ids),
            "format_compliance": bool(first_id in token_ids_for_text(TOKENIZER, format_target)),
        }
        all_rows.append(row)
        free_memory()

all_df = pd.DataFrame(all_rows)
print("\n[COMPUTE COMPLETE] Data collected safely in memory. Proceeding to Cell 2 for saving and plotting...", flush=True)

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import scipy.cluster.hierarchy as sch

# Setup Paths & Styles (Assuming Cell 1 was executed)
OUT_DIR = "/kaggle/working/phi3_prompt_surgery_atlas"
CSV_DIR = os.path.join(OUT_DIR, "csv")
PLOTS_DIR = os.path.join(OUT_DIR, "plots")
REPORTS_DIR = os.path.join(OUT_DIR, "reports")
for p in [OUT_DIR, CSV_DIR, PLOTS_DIR, REPORTS_DIR]: os.makedirs(p, exist_ok=True)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.dpi": 140, "savefig.dpi": 180, "axes.titlesize": 13, "axes.labelsize": 11, "xtick.labelsize": 10, "ytick.labelsize": 10, "legend.fontsize": 9})

# Helper Functions
def save_df(df, path): df.to_csv(path, index=False)
def save_json(obj, path): 
    with open(path, "w") as f: json.dump(obj, f, indent=2)

def annotated_bar(ax, fmt="{:.3f}"):
    for p in ax.patches:
        h = p.get_height()
        if np.isfinite(h): ax.annotate(fmt.format(h), (p.get_x() + p.get_width() / 2.0, h), ha="center", va="bottom", fontsize=8, xytext=(0, 3), textcoords="offset points")

def radar_chart(categories, values_dict, title, path):
    N = len(categories)
    angles = [n / float(N) * 2 * np.pi for n in range(N)]
    angles += angles[:1]
    plt.figure(figsize=(8, 8))
    ax = plt.subplot(111, polar=True); ax.set_theta_offset(np.pi / 2); ax.set_theta_direction(-1)
    plt.xticks(angles[:-1], categories, size=10); ax.set_rlabel_position(0)
    colors = ['#4C72B0', '#C44E52', '#55A868', '#DD8452', '#8172B3']
    for idx, (label, values) in enumerate(values_dict.items()):
        vals = list(values); vals += vals[:1]
        ax.plot(angles, vals, linewidth=2, linestyle='solid', label=label, color=colors[idx % len(colors)])
        ax.fill(angles, vals, alpha=0.15, color=colors[idx % len(colors)])
    plt.title(title, size=15, y=1.1); plt.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
    plt.tight_layout(); plt.savefig(path); plt.close()

def plot_phase_vector_field(variant_df, task, path):
    plt.figure(figsize=(10, 8))
    vdf = variant_df[variant_df["task"] == task]
    if vdf.empty: return
    base = vdf[vdf["recipe"] == "baseline_plain"].iloc[0]
    plt.scatter([base["format_target_margin"]], [base["content_target_margin"]], color='red', marker='*', s=200, label='Baseline', zorder=5)
    for _, r in vdf.iterrows():
        if r["recipe"] == "baseline_plain": continue
        dx = r["format_target_margin"] - base["format_target_margin"]
        dy = r["content_target_margin"] - base["content_target_margin"]
        plt.arrow(base["format_target_margin"], base["content_target_margin"], dx, dy, head_width=0.1, head_length=0.15, fc=plt.cm.viridis(r["prompt_strength"]), ec=plt.cm.viridis(r["prompt_strength"]), alpha=0.8, length_includes_head=True)
        plt.text(r["format_target_margin"], r["content_target_margin"], r["recipe"], fontsize=8)
    plt.xlabel("Format Target Margin (Syntax)"); plt.ylabel("Content Target Margin (Semantics)"); plt.title(f"{task.upper()}: Prompt Surgery Phase-Space Vector Flow")
    plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.savefig(path); plt.close()

def plot_dla_surface(variant_df, task, path):
    vdf = variant_df[variant_df["task"] == task].sort_values("prompt_strength")
    if vdf.empty or len(vdf) < 2: return
    matrix = np.vstack(vdf["content_dla_curve"].map(lambda x: np.array(json.loads(x))).tolist())
    fig = plt.figure(figsize=(12, 8)); ax = fig.add_subplot(111, projection='3d')
    X, Y = np.meshgrid(range(matrix.shape[1]), range(matrix.shape[0]))
    surf = ax.plot_surface(X, Y, matrix, cmap='magma', edgecolor='none', alpha=0.9)
    ax.set_yticks(range(len(vdf))); ax.set_yticklabels(vdf['recipe'].tolist(), rotation=-45, ha='left', fontsize=7)
    ax.set_xlabel('Transformer Layer Depth'); ax.set_ylabel('Prompt Recipe'); ax.set_zlabel('Content DLA Proxy')
    ax.set_title(f"{task.upper()}: 3D Topography of Content DLA")
    fig.colorbar(surf, ax=ax, shrink=0.5, aspect=5); plt.tight_layout(); plt.savefig(path); plt.close()

def plot_dla_dendrogram(variant_df, task, path):
    vdf = variant_df[variant_df["task"] == task]
    if vdf.empty or len(vdf) < 2: return
    matrix = np.vstack(vdf["content_dla_curve"].map(lambda x: np.array(json.loads(x))).tolist())
    if np.std(matrix) < 1e-8: return
    names = vdf["recipe"].tolist()
    plt.figure(figsize=(10, 6))
    Z = sch.linkage(matrix, method='ward', metric='euclidean')
    sch.dendrogram(Z, labels=names, leaf_rotation=45, leaf_font_size=10)
    plt.title(f"{task.upper()}: Hierarchical Clustering of Target DLA Profiles"); plt.ylabel("Euclidean Distance")
    plt.tight_layout(); plt.savefig(path); plt.close()

# Ensure variables from Cell 1 exist
try:
    _ = all_df
except NameError:
    raise RuntimeError("You must execute Cell 1 before running Cell 2.")

print("\n[ANALYSIS] Building Delta Dataframes and Variant Summaries...", flush=True)

# Build Delta DataFrame
base = all_df[all_df["recipe"] == "baseline_plain"].copy()
merge_cols = ["task", "sample_id"]
base = base.rename(columns={c: f"baseline_{c}" for c in base.columns if c not in merge_cols})

rows = []
for _, r in all_df.iterrows():
    if r["recipe"] == "baseline_plain": continue
    b = base[(base["task"] == r["task"]) & (base["sample_id"] == r["sample_id"])]
    if b.empty: continue
    b = b.iloc[0]
    rr = r.to_dict()
    
    # Safe Array Subtraction
    c_dla_r = np.array(json.loads(r["content_dla_curve"]))
    c_dla_b = np.array(json.loads(b["baseline_content_dla_curve"]))
    
    if pd.notna(r.get("format_dla_curve")) and pd.notna(b.get("baseline_format_dla_curve")):
        f_dla_r = np.array(json.loads(r["format_dla_curve"]))
        f_dla_b = np.array(json.loads(b["baseline_format_dla_curve"]))
        delta_f_dla = (f_dla_r - f_dla_b).tolist()
    else:
        delta_f_dla = np.zeros_like(c_dla_r).tolist()
    
    rr.update({
        "baseline_correct": bool(b["baseline_correct"]),
        "baseline_format_target_margin": float(b["baseline_format_target_margin"]),
        "baseline_content_target_margin": float(b["baseline_content_target_margin"]),
        "delta_format_target_margin": float(r["format_target_margin"] - b["baseline_format_target_margin"]),
        "delta_content_target_margin": float(r["content_target_margin"] - b["baseline_content_target_margin"]),
        "delta_content_dla_curve": json.dumps((c_dla_r - c_dla_b).tolist()),
        "delta_format_dla_curve": json.dumps(delta_f_dla),
    })
    rows.append(rr)
delta_df = pd.DataFrame(rows)

# Build Variant Summary
agg_rows = []
for (task, recipe), g in all_df.groupby(["task", "recipe"]):
    base_rows = delta_df[(delta_df["task"] == task) & (delta_df["recipe"] == recipe)] if recipe != "baseline_plain" else pd.DataFrame()
    
    # Safe Matrix Aggregation over Samples
    c_dla = np.mean([json.loads(x) for x in g["content_dla_curve"]], axis=0).tolist()
    
    valid_f = [json.loads(x) for x in g["format_dla_curve"] if pd.notna(x)]
    f_dla = np.mean(valid_f, axis=0).tolist() if valid_f else np.zeros_like(c_dla).tolist()
    
    row = {
        "task": task, "recipe": recipe, "prompt_strength": float(g["prompt_strength"].iloc[0]),
        "accuracy": float(g["correct"].mean()), "format_compliance": float(g["format_compliance"].mean()),
        "content_target_margin": float(g["content_target_margin"].mean()), "format_target_margin": float(g["format_target_margin"].mean()),
        "content_full_entropy": float(g["content_full_entropy"].mean()), "prompt_tokens": float(g["prompt_tokens"].mean()),
        "content_dla_curve": json.dumps(c_dla), "format_dla_curve": json.dumps(f_dla),
    }
    
    if recipe == "baseline_plain":
        row.update({
            "delta_content_target_margin": 0.0, 
            "delta_content_dla_curve": json.dumps(np.zeros_like(c_dla).tolist())
        })
    else:
        d_c_dla = np.mean([json.loads(x) for x in base_rows["delta_content_dla_curve"]], axis=0).tolist() if not base_rows.empty else np.zeros_like(c_dla).tolist()
        d_c_margin = float(base_rows["delta_content_target_margin"].mean()) if not base_rows.empty else 0.0
        row.update({
            "delta_content_target_margin": d_c_margin, 
            "delta_content_dla_curve": json.dumps(d_c_dla)
        })
    agg_rows.append(row)
variant_df = pd.DataFrame(agg_rows)

print("[PLOTTING] Generating Advanced Analytics and DLA Heatmaps...", flush=True)

# 1. Plot DLA Heatmaps per Task
for task in ["gsm8k", "strategyqa", "mmlu"]:
    tdf = variant_df[variant_df["task"] == task].copy().sort_values("prompt_strength")
    if tdf.empty: continue
        
    mat = np.vstack(tdf["delta_content_dla_curve"].map(lambda x: np.array(json.loads(x))).tolist())
    
    fig, ax = plt.subplots(figsize=(13.5, max(5.0, 0.6 * len(tdf))))
    vmax = float(np.nanmax(mat)) if np.isfinite(mat).any() else 0.1
    vmin = float(np.nanmin(mat)) if np.isfinite(mat).any() else -0.1
    if vmin >= 0: vmin = -0.1
    if vmax <= 0: vmax = 0.1
    if vmin == vmax: vmin -= 0.1; vmax += 0.1
        
    norm = TwoSlopeNorm(vmin=vmin, vcenter=0.0, vmax=vmax)
    im = ax.imshow(mat, aspect="auto", cmap="coolwarm", norm=norm)
    ax.set_title(f"{task.upper()} Content DLA Delta vs Baseline")
    ax.set_xlabel("Layer"); ax.set_ylabel("Recipe")
    ax.set_yticks(range(len(tdf))); ax.set_yticklabels(tdf["recipe"].tolist())
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout(); plt.savefig(os.path.join(PLOTS_DIR, f"{task}_content_dla_heatmap.png"), dpi=180); plt.close()

    # Advanced Analysis
    plot_phase_vector_field(variant_df, task, os.path.join(PLOTS_DIR, f"{task}_phase_vector_field.png"))
    plot_dla_surface(variant_df, task, os.path.join(PLOTS_DIR, f"{task}_dla_surface_3d.png"))
    plot_dla_dendrogram(variant_df, task, os.path.join(PLOTS_DIR, f"{task}_dla_dendrogram.png"))

# 2. Multi-Dimensional Radar Fingerprint
radar_data = {}
for idx, v in variant_df.iterrows():
    if v["recipe"] in ["baseline_plain", "verbose_xml", "hash_selfcheck"]: # Sub-sample for visibility
        radar_data[f"{v['task'][:3].upper()}_{v['recipe']}"] = [
            v["accuracy"], v["format_compliance"],
            max(0, 1.0 - (v["content_full_entropy"] / max(1e-12, variant_df["content_full_entropy"].max()))),
            max(0, v["content_target_margin"] / max(1e-12, variant_df["content_target_margin"].max())),
            max(0, 1.0 - (v["prompt_tokens"] / max(1e-12, variant_df["prompt_tokens"].max())))
        ]
if radar_data:
    radar_chart(["Accuracy", "Format", "Entropy (Inv)", "Margin", "Efficiency"], radar_data, "Variant Fingerprints", os.path.join(PLOTS_DIR, "overall_variant_radar.png"))

# 3. Save Final CSVs
save_df(all_df, os.path.join(CSV_DIR, "all_results.csv"))
save_df(variant_df, os.path.join(CSV_DIR, "variant_summary.csv"))

print("\n===== FINISHED! =====", flush=True)
print(f"Data and charts saved successfully in {OUT_DIR}", flush=True)